In [1]:
import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import duckdb as duckdb

C:\Users\perti\OneDrive\Documentos\GIS\Projects\tracklog\.venv\Lib\site-packages\movingpandas\__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


In [2]:
gdf = gpd.read_file('data/2013.gpx', layer="track_points")
filtered_gdf = gdf.loc[(gdf['time'] >= '2013-01-09') & (gdf['time'] < '2013-01-10')]

In [7]:
filtered_gdf

,track_fid,track_seg_id,track_seg_point_id,ele,time,magvar,geoidheight,name,cmt,desc,...,sym,type,fix,sat,hdop,vdop,pdop,ageofdgpsdata,dgpsid,geometry
2603,2,0,0,1122.48,2013-01-09 09:45:55+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.9828 43.27167)
2604,2,0,1,1125.36,2013-01-09 09:46:20+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.98276 43.27167)
2605,2,0,2,1125.84,2013-01-09 09:46:26+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.98273 43.27167)
2606,2,0,3,1125.84,2013-01-09 09:46:29+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.98271 43.27165)
2607,2,0,4,1125.84,2013-01-09 09:46:31+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.98273 43.27164)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3227,2,2,62,299.59,2013-01-09 15:06:26+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-5.09026 43.2095)
3228,2,2,63,302.95,2013-01-09 15:06:42+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-5.09051 43.2094)
3229,2,2,64,295.26,2013-01-09 15:06:53+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-5.09056 43.20959)
3230,2,2,65,294.78,2013-01-09 15:07:09+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-5.09052 43.20978)


In [10]:
mdf = mpd.TrajectoryCollection(filtered_gdf, "track_fid", t="time")
mdf

C:\Users\perti\OneDrive\Documentos\GIS\Projects\tracklog\.venv\Lib\site-packages\movingpandas\trajectory.py:158: TimeZoneWarning: Time zone information dropped from trajectory. All dates and times will use local time. This is applied by doing df.tz_localize(None). To use UTC or a different time zone, convert and drop time zone information prior to trajectory creation.
  warnings.warn(


TrajectoryCollection with 1 trajectories

In [38]:
mdf.add_distance()
mdf.add_speed()


RuntimeError: Trajectory already has a column named distance! Use overwrite=True to overwrite exiting values or update the name arg.

In [44]:
a = mdf.to_point_gdf()
a['slope'] = a['ele'].diff()
a['slope2'] = a['slope']/a['distance']
a.loc[(a.index > "2013-01-09 11:55:30") & (a.index < "2013-01-09 14:52:00")]

,track_fid,track_seg_id,track_seg_point_id,ele,magvar,geoidheight,name,cmt,desc,src,...,hdop,vdop,pdop,ageofdgpsdata,dgpsid,geometry,distance,speed,slope,slope2
time,,,,,,,,,,,,,,,,,,,,,
2013-01-09 11:55:48,2,1,63,1128.24,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-4.98234 43.27171),4.129661,0.206483,0.00,0.000000
2013-01-09 11:56:02,2,1,64,1129.21,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-4.98233 43.27173),2.090974,0.149355,0.97,0.463899
2013-01-09 14:50:58,2,2,0,295.74,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-4.98236 43.27172),2.361572,0.000225,-833.47,-352.930183
2013-01-09 14:51:21,2,2,1,281.32,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-5.09069 43.20963),11180.838064,486.123394,-14.42,-0.001290
2013-01-09 14:51:26,2,2,2,280.84,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-5.09064 43.20968),7.183032,1.436606,-0.48,-0.066824
2013-01-09 14:51:45,2,2,3,279.40,NaN,NaN,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,POINT (-5.09042 43.20979),21.481808,1.130621,-1.44,-0.067033


In [46]:
a['slope2'].describe()

count    628.000000
mean      -0.564543
std       14.094436
min     -352.930183
25%       -0.089840
50%        0.000000
75%        0.116283
max        2.147813
Name: slope2, dtype: float64

In [48]:
gdf

,track_fid,track_seg_id,track_seg_point_id,ele,time,magvar,geoidheight,name,cmt,desc,...,sym,type,fix,sat,hdop,vdop,pdop,ageofdgpsdata,dgpsid,geometry
0,0,0,0,1045.57,2013-01-07 09:56:18+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.985808 43.275991)
1,0,0,1,1045.09,2013-01-07 09:56:34+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.985698 43.27612)
2,0,0,2,1045.09,2013-01-07 09:56:35+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.98569 43.276128)
3,0,0,3,1045.57,2013-01-07 09:56:37+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.985674 43.27614)
4,0,0,4,1046.05,2013-01-07 09:56:38+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-4.985669 43.276144)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50970,81,0,861,NaN,2013-12-21 20:17:20+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-8.105532 43.534285)
50971,81,0,862,NaN,2013-12-21 20:17:32+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-8.105272 43.534326)
50972,81,0,863,NaN,2013-12-21 20:17:44+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-8.105028 43.534267)
50973,81,0,864,NaN,2013-12-21 20:17:55+00:00,NaN,NaN,None,None,None,...,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-8.104849 43.534141)
